<a href="https://colab.research.google.com/github/safarashwan2005/Default-optimized-/blob/main/default_optimized_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import warnings
import numpy as np
import scipy.sparse as sp
from scipy.sparse import kron as sp_kron
import cupy as cp
import cupyx.scipy.sparse as cp_sp
import time
import gc
start_time = time.perf_counter()

warnings.filterwarnings("ignore", category=sp.SparseEfficiencyWarning)



class NVCenterSystemBaseGPU:

    def __init__(self, params):
        self.N = params['N']
        self.D = params['D']
        self.omega_0 = params['omega_0']
        self.gamma_e = params['gamma_e']
        self.Bz = params['Bz']
        self.zeta = params['zeta']
        self.g = params['g']

        # Single-spin operators constructed on CPU to avoid GPU VRAM exhaustion during expansion
        self.I3 = sp.eye(3, dtype=complex, format='csr')
        self.Sz = sp.csr_matrix(
            np.array([[1, 0, 0], [0, 0, 0], [0, 0, -1]], dtype=complex)
        )
        self.Sx = sp.csr_matrix(
            (1 / np.sqrt(2))
            * np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]], dtype=complex)
        )
        self.Sy_prime = sp.csr_matrix(
            (1 / np.sqrt(2))
            * np.array([[0, -1j, 0], [1j, 0, -1j], [0, 1j, 0]], dtype=complex)
        )

    def _expand_operator_sparse(self, op, target_index):
        """Expands a 3x3 operator onto the N-spin Hilbert space in CPU system memory."""
        result = op if target_index == 0 else self.I3
        for i in range(1, self.N):
            current_op = op if i == target_index else self.I3
            result = sp_kron(result, current_op, format='csr')
        return result

    def get_control_hamiltonian(self):
        dim = 3**self.N
        H_control = sp.csr_matrix((dim, dim), dtype=complex)
        cos_z = np.cos(self.zeta)
        sin_z = np.sin(self.zeta)

        for j in range(self.N):
            Sx_j = self._expand_operator_sparse(self.Sx, j)
            Sy_p_j = self._expand_operator_sparse(self.Sy_prime, j)
            H_control = H_control + 0.5 * (cos_z * Sx_j + sin_z * Sy_p_j)

        return H_control

    def get_ground_state(self):
        ms_0 = np.array([0, 1, 0], dtype=complex)
        ground_state = ms_0
        for _ in range(1, self.N):
            ground_state = np.kron(ground_state, ms_0)
        return ground_state

    def _euler_solve_optimized(self, H_drift, H_control, u_array, y0, t_total, num_steps):
        # Executes this integration on GPU.
        dt = t_total / num_steps
        c = -1j
        psi = y0.copy()

        for i in range(num_steps):
            u_t = u_array[i]
            k1 = c * (H_drift.dot(psi) + u_t * H_control.dot(psi))
            psi += dt * k1

        return psi

    def evolve_state_optimized(self, initial_state, t_total, control_pulse, time_steps=2000):
        if not callable(control_pulse):
            raise TypeError("control_pulse must be a callable function (e.g., a lambda).")

        H_drift_cpu = self.get_drift_hamiltonian()
        H_control_cpu = self.get_control_hamiltonian()

        H_drift = cp_sp.csr_matrix(H_drift_cpu)
        H_control = cp_sp.csr_matrix(H_control_cpu)

        initial_state_gpu = cp.asarray(initial_state).reshape(-1, 1)

        # Generate timestamps for Euler (1 per step)
        euler_time_points = cp.linspace(0, t_total, time_steps, endpoint=False)

        t_np = cp.asnumpy(euler_time_points)
        u_array = cp.asarray(control_pulse(t_np))

        return self._euler_solve_optimized(
            H_drift, H_control, u_array, initial_state_gpu, t_total, num_steps=time_steps
        )
class NVCenterNearestNeighborGPU(NVCenterSystemBaseGPU):

    def get_drift_hamiltonian(self):
        dim = 3**self.N
        H_drift = sp.csr_matrix((dim, dim), dtype=complex)

        Sz_sq = self.Sz.dot(self.Sz)

        for j in range(self.N):
            Sz_j_sq = self._expand_operator_sparse(Sz_sq, j)
            Sz_j = self._expand_operator_sparse(self.Sz, j)

            term1 = (self.D[j] - self.omega_0) * Sz_j_sq
            term2 = self.gamma_e * self.Bz[j] * Sz_j
            H_drift = H_drift + (term1 - term2)

        for j in range(self.N - 1):
            Sz_j = self._expand_operator_sparse(self.Sz, j)
            Sz_j_next = self._expand_operator_sparse(self.Sz, j + 1)
            H_drift = H_drift + self.g[j] * Sz_j.dot(Sz_j_next)

        return H_drift


# =======================================
# Execution Block
# ==========================================
if __name__ == "__main__":
    NUM_RUNS = 10
    n_range = range(1, 15)


    times_per_N = {N: [] for N in n_range}

    for run in range(1, NUM_RUNS + 1):
        print(f"\n========================================")
        print(f"             STARTING RUN {run}/{NUM_RUNS}             ")
        print(f"========================================")

        for N_test in n_range:

            iter_start_time = time.perf_counter()

            print(f"\n{'='*40}")
            print(f" [RUN {run}/{NUM_RUNS}] SIMULATING FOR N = {N_test} NV CENTERS ")
            print(f"{'='*40}")

            test_params = {
                'N': N_test,
                'D': [2870.0] * N_test,
                'omega_0': 2870.0,
                'gamma_e': 28000.0,
                'Bz': [1e-3] * N_test,
                'zeta': np.pi / 4,
                'g': [0.1] * (N_test - 1),
            }

            print("Building Sparse Hamiltonians on CPU...")
            nv_chain = NVCenterNearestNeighborGPU(test_params)
            initial_state_cpu = nv_chain.get_ground_state()
            t_pulse = 1.0

            print("--- Testing Lambda Function Pulse ---")
            lambda_pulse = lambda t: 2.0 * np.sin(np.pi * (t / t_pulse))

            final_state_lambda = nv_chain.evolve_state_optimized(
                initial_state=initial_state_cpu,
                t_total=t_pulse,
                control_pulse=lambda_pulse,
                time_steps=2000,
            )

            norm_lambda = cp.asnumpy(cp.linalg.norm(final_state_lambda))
            print("Norm check (Lambda):", norm_lambda)

            cp.cuda.Stream.null.synchronize()
            iter_end_time = time.perf_counter()
            elapsed_time = iter_end_time - iter_start_time


            times_per_N[N_test].append(elapsed_time)

            print(f"\n[TIME TAKEN FOR N = {N_test} (Run {run})]: {elapsed_time:.4f} seconds")

            # Free GPU memory and CPU garbage collection
            del nv_chain, initial_state_cpu, final_state_lambda
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()

    end_time = time.perf_counter()

    print(f"\n{'='*50}")
    print(f"TOTAL EXECUTION TIME (ALL {NUM_RUNS} RUNS): {end_time - start_time:.4f} seconds")
    print(f"{'='*50}")

    print(f"\n{'='*50}")
    print(f" AVERAGE EXECUTION TIME PER N OVER {NUM_RUNS} RUNS ")
    print(f"{'='*50}")
    for N_test in n_range:
        avg_seconds = np.mean(times_per_N[N_test])
        std_seconds = np.std(times_per_N[N_test])
        print(f"N = {N_test:2d} | Avg Time: {avg_seconds:.4f} s (±{std_seconds:.4f} s)")
    print(f"{'='*50}")




             STARTING RUN 1/10             

 [RUN 1/10] SIMULATING FOR N = 1 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function Pulse ---
Norm check (Lambda): 1.0000056641053718

[TIME TAKEN FOR N = 1 (Run 1)]: 2.2105 seconds

 [RUN 1/10] SIMULATING FOR N = 2 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function Pulse ---
Norm check (Lambda): 1.000011328377983

[TIME TAKEN FOR N = 2 (Run 1)]: 0.9728 seconds

 [RUN 1/10] SIMULATING FOR N = 3 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function Pulse ---
Norm check (Lambda): 1.000016992813091

[TIME TAKEN FOR N = 3 (Run 1)]: 0.6245 seconds

 [RUN 1/10] SIMULATING FOR N = 4 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function Pulse ---
Norm check (Lambda): 1.0000226574107114

[TIME TAKEN FOR N = 4 (Run 1)]: 0.6470 seconds

 [RUN 1/10] SIMULATING FOR N = 5 NV CENTERS 
Building Sparse Hamiltonians on CPU...
--- Testing Lambda Function 